# 面试题：RAG 里的关键词检索应该怎样设计，如何从零实现并证明答案有依据？

## 面试回答主线

关键词检索不是简单字符串包含，而是一条可审计管线：字段级分析、倒排索引、BM25/BM25F 打分、精确标识符加权、ACL/版本过滤、候选去重和上下文预算。RAG 中它特别擅长错误码、产品型号、法规条款和专有名词，通常与向量召回并行而不是互相替代。过滤必须发生在候选进入生成上下文之前，不能先把无权限文档交给模型再要求它忽略。评估要拆成 retrieval recall、排序指标、引用支持率与拒答率；只看最终语言流畅度会掩盖召回失败。低分或无证据查询应拒答，而不是强行让 top-1 文档支持答案。

## 真实案例：支付与账户帮助中心

八篇文档包含公开/内部 ACL、active/retired 版本、错误码与不同支付渠道。六条有答案查询和一条域外查询都带人工 gold；内容是脱敏教学数据，时效规则不应外推真实业务。

In [1]:
import math  # 导入对数以手写 BM25 的 IDF 与长度归一化。
import re  # 导入正则表达式以识别 ASCII 标识符和中文词片。
from collections import Counter, defaultdict  # 导入计数器和倒排表默认容器。
documents = [  # 构造带字段、权限与版本状态的帮助中心文档。
    {"id": "D1", "title": "微信退款到账时间", "body": "微信支付退款通常一到三个工作日到账，可在订单详情查看进度。", "tags": "微信 退款 进度", "acl": "public", "status": "active"},  # 公开有效的微信退款说明。
    {"id": "D2", "title": "信用卡退款周期", "body": "信用卡退款由发卡行处理，通常三到十五个工作日到账。", "tags": "信用卡 退款 到账", "acl": "public", "status": "active"},  # 公开有效的信用卡时效。
    {"id": "D3", "title": "错误码 E102 验证码失败", "body": "E102 表示验证码发送受限，请等待十分钟后重试并检查手机号。", "tags": "E102 验证码 登录", "acl": "public", "status": "active"},  # 错误码精确匹配文档。
    {"id": "D4", "title": "E102 设备绑定内部排查", "body": "内部日志遇到 E102 时检查设备指纹和风控名单。", "tags": "E102 设备 风控", "acl": "internal", "status": "active"},  # 同错误码但禁止公开用户访问。
    {"id": "D5", "title": "SAVE20 优惠券规则", "body": "SAVE20 仅在活动期和指定商品可用，过期或品类不符会显示失效。", "tags": "SAVE20 优惠券 失效", "acl": "public", "status": "active"},  # 含精确营销代码的规则文档。
    {"id": "D6", "title": "永久删除账户", "body": "在隐私设置提交删除申请，完成身份验证后进入七天冷静期。", "tags": "账户 删除 注销 隐私", "acl": "public", "status": "active"},  # 账号删除流程文档。
    {"id": "D7", "title": "包裹物流查询", "body": "在订单物流页查看承运商、运单号和最新配送轨迹。", "tags": "包裹 物流 运单", "acl": "public", "status": "active"},  # 包裹查询公开文档。
    {"id": "D8", "title": "旧版退款规则", "body": "退款退款退款进度进度进度进度进度进度曾统一承诺当天到账，此规则已经废止。", "tags": "退款 进度 到账 旧版", "acl": "public", "status": "retired"},  # 重复关键词会诱骗基线但必须被版本过滤。
]  # 结束帮助中心文档列表。
queries = [  # 构造覆盖渠道、错误码、营销码、账户与否定意图的查询。
    {"id": "Q1", "text": "信用卡退款多久到账", "gold": "D2"},  # 渠道限定应命中信用卡文档。
    {"id": "Q2", "text": "E102 验证码收不到怎么办", "gold": "D3"},  # 精确错误码与验证码共同约束。
    {"id": "Q3", "text": "SAVE20 优惠券为什么失效", "gold": "D5"},  # 精确营销代码查询。
    {"id": "Q4", "text": "怎么永久删除账户", "gold": "D6"},  # 隐私删除流程查询。
    {"id": "Q5", "text": "微信退款进度在哪看", "gold": "D1"},  # 微信渠道与进度组合查询。
    {"id": "Q6", "text": "不想退款，只想查包裹物流", "gold": "D7"},  # 否定退款意图后应检索物流。
]  # 结束带人工相关文档的查询列表。
print("文档  ACL/status          title")  # 输出文档集合预览标题。
for document in documents:  # 逐条展示索引前的文档元数据。
    print(f"{document['id']:<4} {document['acl'] + '/' + document['status']:<18} {document['title']}")  # 输出权限、版本和标题。
print("查询与人工 gold：", [(query["id"], query["text"], query["gold"]) for query in queries])  # 输出完整评测查询以明确目标。

文档  ACL/status          title
D1   public/active      微信退款到账时间
D2   public/active      信用卡退款周期
D3   public/active      错误码 E102 验证码失败
D4   internal/active    E102 设备绑定内部排查
D5   public/active      SAVE20 优惠券规则
D6   public/active      永久删除账户
D7   public/active      包裹物流查询
D8   public/retired     旧版退款规则
查询与人工 gold： [('Q1', '信用卡退款多久到账', 'D2'), ('Q2', 'E102 验证码收不到怎么办', 'D3'), ('Q3', 'SAVE20 优惠券为什么失效', 'D5'), ('Q4', '怎么永久删除账户', 'D6'), ('Q5', '微信退款进度在哪看', 'D1'), ('Q6', '不想退款，只想查包裹物流', 'D7')]


## Baseline（基线）：全字段 substring 次数，不做 ACL 与版本过滤

基线把查询中出现的连续中文双字片段和 ASCII 串拿去数原文出现次数，重复“退款”的旧文档会获益，内部 E102 文档也可能进入结果。它没有字段权重、IDF、否定或拒答门槛。

In [2]:
def baseline_terms(text):  # 用连续中文双字片段和 ASCII 串构造朴素查询词。
    chinese_runs = re.findall(r"[\u4e00-\u9fff]+", text)  # 提取连续中文片段。
    chinese_bigrams = [run[index : index + 2] for run in chinese_runs for index in range(max(0, len(run) - 1))]  # 为每个中文片段生成重叠双字词。
    ascii_terms = [term.lower() for term in re.findall(r"[A-Za-z]+\d*", text)]  # 提取并小写英文或错误码。
    return chinese_bigrams + ascii_terms  # 返回不含语言理解的朴素词列表。
def baseline_search(query, top_k=3):  # 对全部文档执行 substring 次数排序。
    terms = baseline_terms(query)  # 分析查询得到双字片段与 ASCII 标识符。
    scored = []  # 创建列表保存每篇文档的累计包含次数。
    for document in documents:  # 错误基线故意遍历公开、内部和废止文档。
        haystack = " ".join([document["title"], document["body"], document["tags"]]).lower()  # 合并所有字段且丢失字段边界。
        score = sum(haystack.count(term) for term in terms)  # 把每个查询词的原文次数直接相加。
        scored.append((document["id"], score))  # 保存文档编号和朴素得分。
    return sorted(scored, key=lambda item: (-item[1], item[0]))[:top_k]  # 用得分和文档编号稳定排序。
baseline_hits = 0  # 统计朴素检索 top-1 命中数量。
print("查询  baseline top3                 gold  top1正确")  # 输出逐查询基线结果表标题。
for query in queries:  # 在全部人工评测查询上运行相同基线。
    ranking = baseline_search(query["text"])  # 取得当前查询的前三文档及分数。
    is_correct = ranking[0][0] == query["gold"]  # 判断 top-1 是否命中人工 gold。
    baseline_hits += int(is_correct)  # 累加正确查询数。
    print(f"{query['id']:<4} {str(ranking):<31} {query['gold']:<4} {is_correct}")  # 输出排名、gold 和命中状态。
baseline_accuracy = baseline_hits / len(queries)  # 计算朴素关键词 top-1 准确率。
print(f"朴素 substring top-1：{baseline_accuracy:.1%}")  # 输出后续 BM25F 管线的同数据基线。

查询  baseline top3                 gold  top1正确
Q1   [('D2', 13), ('D8', 7), ('D1', 5)] D2   True
Q2   [('D3', 9), ('D4', 3), ('D6', 1)] D3   True
Q3   [('D5', 9), ('D1', 0), ('D2', 0)] D5   True
Q4   [('D6', 8), ('D1', 0), ('D2', 0)] D6   True
Q5   [('D8', 13), ('D1', 9), ('D2', 3)] D1   False
Q6   [('D7', 6), ('D8', 5), ('D1', 3)] D7   True
朴素 substring top-1：83.3%


## 核心实现一：领域 analyzer、字段倒排表与 BM25F

领域词典用最长匹配保留“信用卡、验证码、优惠券”等业务词，ASCII 标识符统一大写。倒排表按字段保存 TF 和长度，BM25F 先做字段长度归一化与权重合并，再乘全局 IDF。ACL 和 active 状态在打分前过滤。

In [3]:
domain_lexicon = ["信用卡", "退款", "到账", "验证码", "手机号", "优惠券", "失效", "永久", "删除", "账户", "微信", "进度", "包裹", "物流", "运单", "查询", "活动期", "商品", "设置", "申请"]  # 定义帮助中心领域词典。
domain_lexicon = sorted(domain_lexicon, key=lambda token: (-len(token), token))  # 按长度降序保证最长业务词优先。
stop_words = {"怎么", "为什么", "多久", "在哪", "怎么办", "只想", "我", "的", "了"}  # 定义不参与排序的常见查询词。
def analyze(text):  # 对中文业务文本执行最长匹配与 ASCII 标识符切分。
    normalized = text.upper()  # 统一错误码和营销代码的 ASCII 大小写。
    tokens = []  # 创建列表保存规范化后的检索 token。
    index = 0  # 从文本首字符开始扫描。
    while index < len(normalized):  # 持续扫描直到消费完整文本。
        if normalized[index].isspace() or normalized[index] in "，。！？、：；,-":  # 空白和普通标点只作为边界。
            index += 1  # 跳过当前边界字符。
            continue  # 回到循环顶部处理下一位置。
        identifier = re.match(r"[A-Z]+\d+", normalized[index:])  # 尝试识别错误码或营销代码。
        if identifier is not None:  # 精确 ASCII 标识符应保持完整。
            tokens.append(identifier.group(0))  # 保存完整标识符 token。
            index += len(identifier.group(0))  # 跨过已经消费的标识符。
            continue  # 回到循环顶部继续扫描。
        matched = next((token for token in domain_lexicon if normalized.startswith(token, index)), None)  # 在当前位置寻找最长领域词。
        if matched is not None:  # 找到领域词时保留完整业务语义。
            if matched not in stop_words:  # 停用词不进入倒排索引。
                tokens.append(matched)  # 保存当前领域 token。
            index += len(matched)  # 跳过领域词覆盖的字符。
            continue  # 回到循环顶部继续扫描。
        if "\u4e00" <= normalized[index] <= "\u9fff":  # 未登录汉字使用单字回退避免整段丢失。
            tokens.append(normalized[index])  # 保存当前单字 token。
        index += 1  # 向后移动一个字符保证扫描推进。
    return tokens  # 返回确定性的领域 token 序列。
field_weights = {"title": 2.4, "body": 1.0, "tags": 1.8}  # 设置标题、正文和标签的业务权重。
field_b = {"title": 0.3, "body": 0.75, "tags": 0.2}  # 设置各字段的长度归一化强度。
postings = {field: defaultdict(dict) for field in field_weights}  # 创建字段到 term 再到 doc TF 的倒排结构。
field_lengths = {field: {} for field in field_weights}  # 保存每篇文档各字段的 token 长度。
document_frequencies = Counter()  # 统计 term 出现过的不同文档数。
for document in documents:  # 遍历全部文档构建可审计倒排索引。
    seen_terms = set()  # 创建当前文档已经计入 DF 的 term 集合。
    for field in field_weights:  # 分字段分析标题、正文和标签。
        field_tokens = analyze(document[field])  # 使用统一 analyzer 得到字段 token。
        field_lengths[field][document["id"]] = len(field_tokens)  # 保存当前字段长度。
        for term, frequency in Counter(field_tokens).items():  # 遍历字段内每个 term 的 TF。
            postings[field][term][document["id"]] = frequency  # 写入字段级倒排 posting。
            seen_terms.add(term)  # 标记该文档包含当前 term。
    for term in seen_terms:  # 每个文档对同一 term 的 DF 只贡献一次。
        document_frequencies[term] += 1  # 累加全字段文档频次。
average_lengths = {field: sum(lengths.values()) / len(lengths) for field, lengths in field_lengths.items()}  # 计算各字段平均长度。
print("analyzer 示例：", analyze("E102 验证码收不到，SAVE20为什么失效"))  # 输出错误码、中文词和营销码的实际切分。
print("字段平均长度：", {field: round(value, 2) for field, value in average_lengths.items()})  # 输出 BM25F 长度归一化基准。
print("‘退款’ postings：", {field: dict(postings[field].get("退款", {})) for field in field_weights})  # 输出字段级 TF 倒排明细。

analyzer 示例： ['E102', '验证码', '收', '不', '到', 'SAVE20', '为', '什', '么', '失效']
字段平均长度： {'title': 5.0, 'body': 21.0, 'tags': 4.0}
‘退款’ postings： {'title': {'D1': 1, 'D2': 1, 'D8': 1}, 'body': {'D1': 1, 'D2': 1, 'D8': 3}, 'tags': {'D1': 1, 'D2': 1, 'D8': 1}}


## 核心实现二：查询否定、精确 ID boost 与逐项解释

`不想退款` 被识别为否定约束，从正向查询词中移除“退款”。精确错误码或营销码在标题/标签完整出现时额外加分。返回值包含每个 term 的 IDF、字段归一化 TF 和最终贡献，便于线上解释与调参。

In [4]:
def analyze_query(text):  # 解析正向检索词、否定词和精确标识符。
    terms = analyze(text)  # 先复用文档 analyzer 得到规范化 token。
    negated = set()  # 创建否定业务词集合。
    if "不想退款" in text:  # 识别教学案例中的明确退款否定短语。
        negated.add("退款")  # 标记退款不应成为正向召回证据。
    positive_terms = [term for term in terms if term not in negated and term not in stop_words]  # 删除否定词和停用词。
    identifiers = {term for term in positive_terms if re.fullmatch(r"[A-Z]+\d+", term)}  # 提取需要精确加权的标识符。
    return positive_terms, negated, identifiers  # 返回查询规划的三个组成部分。
def bm25f_search(text, allowed_acl={"public"}, top_k=3, explain=False):  # 执行 ACL/版本过滤后的 BM25F 排序。
    terms, negated, identifiers = analyze_query(text)  # 解析查询正向词、否定词和精确 ID。
    allowed_documents = [document for document in documents if document["acl"] in allowed_acl and document["status"] == "active"]  # 在打分前应用权限与版本门禁。
    allowed_ids = {document["id"] for document in allowed_documents}  # 创建可参与候选排序的文档编号集合。
    scores = defaultdict(float)  # 创建文档累计 BM25F 分数字典。
    explanations = defaultdict(list)  # 创建文档逐 term 分项解释账本。
    k1 = 1.2  # 设置 BM25 饱和参数控制 TF 边际收益。
    total_documents = len(documents)  # 使用完整索引文档数计算稳定全局 IDF。
    for term in terms:  # 逐个查询 token 累加字段化贡献。
        document_frequency = document_frequencies.get(term, 0)  # 读取该 term 出现过的文档数。
        idf = math.log(1.0 + (total_documents - document_frequency + 0.5) / (document_frequency + 0.5))  # 计算始终为正的 BM25 IDF。
        candidate_ids = set().union(*(postings[field].get(term, {}).keys() for field in field_weights))  # 合并该 term 在各字段的候选文档。
        for document_id in sorted(candidate_ids & allowed_ids):  # 只对有权限且 active 的候选计分。
            weighted_tf = 0.0  # 初始化 BM25F 的字段归一化 TF 合计。
            for field, weight in field_weights.items():  # 逐字段读取 TF、长度与业务权重。
                term_frequency = postings[field].get(term, {}).get(document_id, 0)  # 读取当前字段内 term 次数。
                length_ratio = field_lengths[field][document_id] / average_lengths[field]  # 计算字段长度相对平均值。
                normalized_tf = term_frequency / (1.0 - field_b[field] + field_b[field] * length_ratio)  # 执行字段级长度归一化。
                weighted_tf += weight * normalized_tf  # 按字段业务权重合并 TF。
            bm25_contribution = idf * (weighted_tf * (k1 + 1.0)) / (weighted_tf + k1) if weighted_tf > 0 else 0.0  # 应用 TF 饱和得到 term 贡献。
            document = next(document for document in allowed_documents if document["id"] == document_id)  # 取得候选原文以检查精确标识符。
            exact_boost = 2.5 if term in identifiers and term in (document["title"] + " " + document["tags"]).upper() else 0.0  # 为标题或标签完整 ID 命中加固定分。
            contribution = bm25_contribution + exact_boost  # 合并统计相关性与精确业务加权。
            scores[document_id] += contribution  # 累加到候选文档总分。
            explanations[document_id].append((term, round(idf, 3), round(weighted_tf, 3), round(exact_boost, 3), round(contribution, 3)))  # 保存可读打分分项。
    ranking = sorted(scores.items(), key=lambda item: (-item[1], item[0]))[:top_k]  # 按分数降序和编号稳定返回 top-k。
    return (ranking, explanations, terms, negated) if explain else ranking  # 根据调用方需求返回排名或完整解释。
focus_ranking, focus_explanations, focus_terms, focus_negated = bm25f_search("E102 验证码收不到怎么办", explain=True)  # 对错误码查询执行可解释检索。
print("焦点查询正向 terms：", focus_terms, "，否定 terms：", sorted(focus_negated))  # 输出查询规划结果。
print("焦点查询 ranking：", [(document_id, round(score, 3)) for document_id, score in focus_ranking])  # 输出经过 ACL 和版本过滤的排名。
for document_id, _ in focus_ranking:  # 逐候选展示 BM25F 的真实分项。
    print(document_id, "term/idf/weighted_tf/exact_boost/contribution ->", focus_explanations[document_id])  # 输出逐 term 解释账本。

焦点查询正向 terms： ['E102', '验证码', '收', '不', '到', '怎', '么', '办'] ，否定 terms： []
焦点查询 ranking： [('D3', 7.939), ('D5', 1.792), ('D2', 0.983)]
D3 term/idf/weighted_tf/exact_boost/contribution -> [('E102', 1.281, 4.943, 2.5, 4.768), ('验证码', 1.792, 4.943, 0.0, 3.172)]
D5 term/idf/weighted_tf/exact_boost/contribution -> [('不', 1.792, 1.0, 0.0, 1.792)]
D2 term/idf/weighted_tf/exact_boost/contribution -> [('到', 0.944, 1.077, 0.0, 0.983)]


## 结果表：同一查询集比较 top-1 与安全门禁

这里的指标是六条教学查询的 top-1 accuracy，不冒充线上收益。除了 gold 命中，还打印返回文档的 ACL/status，确保“相关但无权限”不算成功。

In [5]:
bm25_hits = 0  # 统计 BM25F 在六条查询上的 top-1 命中数量。
bm25_results = {}  # 保存逐查询排名供上下文构建和测试复用。
print("查询  baseline top1  BM25F top3                  gold  安全top1")  # 输出统一评测结果表标题。
for query in queries:  # 在与基线相同的查询集合上评估完整管线。
    baseline_top = baseline_search(query["text"])[0][0]  # 取得朴素 substring 的 top-1 文档。
    ranking = bm25f_search(query["text"])  # 执行带过滤、字段权重和否定的 BM25F。
    bm25_results[query["id"]] = ranking  # 保存当前查询的完整 top-k 排名。
    top_document = ranking[0][0] if ranking else None  # 对无召回情况安全取得 top-1。
    is_correct = top_document == query["gold"]  # 判断安全 top-1 是否命中人工 gold。
    bm25_hits += int(is_correct)  # 累加正确查询数。
    rank_view = [(document_id, round(score, 2)) for document_id, score in ranking]  # 格式化候选分数便于阅读。
    print(f"{query['id']:<4} {baseline_top:<14} {str(rank_view):<28} {query['gold']:<4} {is_correct}")  # 输出两种 top-1、完整排名和真值。
bm25_accuracy = bm25_hits / len(queries)  # 计算完整检索管线 top-1 accuracy。
retrieved_ids = {document_id for ranking in bm25_results.values() for document_id, _ in ranking}  # 汇总所有进入候选集的文档编号。
print(f"top-1 对照：substring={baseline_accuracy:.1%}，BM25F+门禁={bm25_accuracy:.1%}")  # 输出相同数据和指标下的提升。
print("任何查询是否泄露 internal/retired：", bool(retrieved_ids.intersection({"D4", "D8"})))  # 输出 ACL 与版本门禁的全局审计结论。

查询  baseline top1  BM25F top3                  gold  安全top1
Q1   D2             [('D2', 6.43), ('D1', 3.22)] D2   True
Q2   D3             [('D3', 7.94), ('D5', 1.79), ('D2', 0.98)] D3   True
Q3   D5             [('D5', 11.62)]              D5   True
Q4   D6             [('D6', 9.05)]               D6   True
Q5   D8             [('D1', 8.77), ('D7', 2.05), ('D2', 1.71)] D1   True
Q6   D7             [('D7', 7.11), ('D5', 1.79), ('D3', 0.69)] D7   True
top-1 对照：substring=83.3%，BM25F+门禁=100.0%
任何查询是否泄露 internal/retired： False


## 结果解读与带引用上下文

字段权重让标题和标签的高精度命中超过正文偶然重复，IDF 让 E102、SAVE20 等稀有词更有区分度，版本/ACL 门禁彻底排除 D4 与 D8。RAG 下一步不是把 top-k 原文无限拼接，而是在预算内保留文档 id、分数和正文，让答案能回指证据。

In [6]:
document_by_id = {document["id"]: document for document in documents}  # 创建文档编号到完整记录的查找表。
def build_context(ranking, character_budget=90):  # 在字符预算内构建带文档 id 的 RAG 上下文。
    chunks = []  # 创建列表保存被预算接纳的证据块。
    used = 0  # 记录已经占用的近似字符预算。
    for document_id, score in ranking:  # 按检索分数顺序处理候选文档。
        document = document_by_id[document_id]  # 取得当前候选的标题和正文。
        chunk = f"[{document_id}] {document['title']}：{document['body']}"  # 为证据添加稳定引用编号。
        if used + len(chunk) > character_budget and chunks:  # 预算已满且至少有一块证据时停止追加。
            break  # 保留更高分证据并结束打包。
        chunks.append(chunk)  # 接纳当前高分证据块。
        used += len(chunk)  # 累加近似上下文字符成本。
    return chunks, used  # 返回有序证据块及实际成本。
def grounded_answer(query_text, ranking, minimum_score=0.8):  # 用检索证据生成可审计的教学答案或拒答。
    if not ranking or ranking[0][1] < minimum_score:  # 没有候选或最高分不足时拒绝无依据回答。
        return "证据不足，建议转人工或补充产品/错误码信息。", []  # 返回明确拒答和空引用列表。
    top_document = document_by_id[ranking[0][0]]  # 取得最高分且已通过门禁的文档。
    first_sentence = top_document["body"].split("。")[0]  # 只抽取第一条受支持事实避免扩写。
    answer = f"{first_sentence}。[{top_document['id']}]"  # 把文档编号附在事实后形成行内引用。
    return answer, [top_document["id"]]  # 返回答案文本和结构化引用。
print("查询  上下文字符  引用  教学答案")  # 输出逐查询 RAG 上下文与答案表标题。
answer_citations = {}  # 保存每个查询实际引用的文档编号。
for query in queries:  # 对六条有答案查询执行上下文构建与 grounded answer。
    ranking = bm25_results[query["id"]]  # 读取已经评估过的安全检索排名。
    chunks, used = build_context(ranking)  # 在固定预算内拼接带引用证据。
    answer, citations = grounded_answer(query["text"], ranking)  # 只依据最高分文档构造教学答案。
    answer_citations[query["id"]] = citations  # 保存引用列表供支持率检查。
    print(f"{query['id']:<4} {used:>10} {str(citations):<7} {answer}")  # 输出成本、引用和有依据答案。

查询  上下文字符  引用  教学答案
Q1           81 ['D2']  信用卡退款由发卡行处理，通常三到十五个工作日到账。[D2]
Q2           51 ['D3']  E102 表示验证码发送受限，请等待十分钟后重试并检查手机号。[D3]
Q3           51 ['D5']  SAVE20 仅在活动期和指定商品可用，过期或品类不符会显示失效。[D5]
Q4           39 ['D6']  在隐私设置提交删除申请，完成身份验证后进入七天冷静期。[D6]
Q5           78 ['D1']  微信支付退款通常一到三个工作日到账，可在订单详情查看进度。[D1]
Q6           86 ['D7']  在订单物流页查看承运商、运单号和最新配送轨迹。[D7]


## 失败案例：强制 top-1 回答域外问题，以及否定词误召回

未知产品码 `ZXQ999` 在本知识库没有证据。错误管线仍拿排序第一篇文档生成话术；修复后最高分为空或低于阈值，系统明确拒答。另一个反例是“不想退款，只查物流”：不处理否定时“退款”会污染候选，查询规划移除否定 term 后才能稳定命中物流文档。

In [7]:
out_of_domain_query = "ZXQ999"  # 构造知识库明确没有收录的未知产品码查询。
out_of_domain_ranking = bm25f_search(out_of_domain_query)  # 用相同安全检索器查找证据。
forced_document = document_by_id["D1"]  # 模拟错误系统在空召回时仍拿固定第一篇文档。
forced_answer = f"{forced_document['body'].split('。')[0]}。[{forced_document['id']}]"  # 构造与房贷无关但语法流畅的错误回答。
safe_answer, safe_citations = grounded_answer(out_of_domain_query, out_of_domain_ranking)  # 使用最低证据分门槛生成安全结果。
raw_negation_terms = analyze("不想退款，只想查包裹物流")  # 展示未做查询规划时的原始 analyzer token。
planned_terms, planned_negated, planned_ids = analyze_query("不想退款，只想查包裹物流")  # 执行否定感知的查询规划。
negation_ranking = bm25f_search("不想退款，只想查包裹物流")  # 对修复后的真实查询执行排序。
internal_baseline_leak = baseline_search("E102 设备风控", top_k=3)  # 用无 ACL 基线检索内部特征明显的查询。
safe_internal_ranking = bm25f_search("E102 设备风控", top_k=3)  # 用权限门禁后的 BM25F 检索同一查询。
print("域外检索 ranking：", out_of_domain_ranking)  # 输出没有任何正分证据的实际结果。
print("错误强制回答：", forced_answer)  # 展示无证据却强行生成的错误行为。
print("门槛修复回答：", safe_answer, "，引用：", safe_citations)  # 展示拒答与空引用修复。
print("否定处理：", raw_negation_terms, "-> 正向", planned_terms, "否定", sorted(planned_negated), "top1", negation_ranking[0][0])  # 展示退款 term 被移除后的物流命中。
print("ACL 对照：baseline", internal_baseline_leak, "-> safe", safe_internal_ranking)  # 展示内部 D4 在安全检索中被排除。

域外检索 ranking： []
错误强制回答： 微信支付退款通常一到三个工作日到账，可在订单详情查看进度。[D1]
门槛修复回答： 证据不足，建议转人工或补充产品/错误码信息。 ，引用： []
否定处理： ['不', '想', '退款', '只', '想', '查', '包裹', '物流'] -> 正向 ['不', '想', '只', '想', '查', '包裹', '物流'] 否定 ['退款'] top1 D7
ACL 对照：baseline [('D4', 8), ('D3', 3), ('D1', 0)] -> safe [('D3', 4.767550798059822)]


## 生产差距与落地清单

教学实现只有八篇文档，中文 analyzer 依赖小词典。线上需要版本化 analyzer、同义词与停用词，支持增量 postings、删除 tombstone、租户/ACL bitmap、短语位置、拼写纠错和查询超时；BM25 候选再与向量召回做 RRF 或学习排序。离线按时间切分评估 recall@k、MRR/nDCG、ACL 泄露、引用支持率和拒答；线上监控零结果率、候选数、索引延迟、旧版本命中、context token 成本及按查询类型拆分的反馈。

## 最小回归测试

断言只保护 gold 命中、权限、否定、引用和拒答合同；逐 term 分项、结果表和失败输出才是设计依据。

In [8]:
assert bm25_accuracy > baseline_accuracy  # 验证完整关键词管线在同一人工查询集上优于 substring 基线。
assert all(bm25_results[query["id"]][0][0] == query["gold"] for query in queries)  # 验证六条教学查询 top-1 全部命中人工 gold。
assert not retrieved_ids.intersection({"D4", "D8"})  # 验证 internal 与 retired 文档从未进入生成候选。
assert "退款" not in planned_terms and "退款" in planned_negated  # 验证否定查询规划不会把退款当正向证据。
assert answer_citations["Q2"] == ["D3"]  # 验证错误码答案引用公开验证码文档而非内部排查文档。
assert out_of_domain_ranking == [] and safe_citations == []  # 验证域外查询无候选且不会伪造引用。
assert "证据不足" in safe_answer  # 验证低证据路径返回明确拒答而不是强制回答。
print("最小回归测试通过：BM25F 相关性、ACL/版本门禁、否定规划、引用与拒答均符合预期。")  # 输出完整顺序执行成功的明确结论。

最小回归测试通过：BM25F 相关性、ACL/版本门禁、否定规划、引用与拒答均符合预期。
